# Corpus Preparation: Manifesto Extraction & Preprocessing Pipeline

In this file i'm building a sentence-level preprocessing pipeline for the UK party manifesto corpus (2005–2024). These are used in my SBERT-based lexicon expansion (also in this directory) and also the stratified corpus sampler to pullout both the weak and gold labels. The below code is setup to pull raw text from the manifesto PDFs. It then applies a (very long) series of targeted OCR repair operations and then segments everything into sentence-level format.

I'm working at sentence level deliberately (not quasi-lvl or whole-doc analysis) because SBERT needs that format as an input to do meaningful & reasonable semantic search in the lexical neighbourhood and it aligns with what I'm measuring. I store each sentence stored alongside its manifesto metadata which means I can audit what was actually corrected at any stage.

## Use of Generative AI
When writing these functions below, I sometimes used generative AI to help me (specifically, Claude Sonnet 4.6 by Anthropic). The way I used it was at a function level, where I asked it for help on implementing and refining specific snippets for the many extraction issues I got. These snippets were mainly concentrated on regex patterns and text repitition/reversal detection, and also included the edge cases I found from the tricky font encodings in the manifestos (like tables of contents and numerical tables). All of my prompting was done on a chat-by-chat basis where I identified a specific extraction/cleanup challenge per query, giving it the relevant context and the behaviour I expected. I also used it to help test the validation functions by giving it the tests I came up with and getting it pick holes in them and identify tricky edge cases that I missed so I could make sure my testing was thorough.

However, all of the pipeline, including the choice of extraction method, the patterns I targeted for detection and repair and all the validation and logging logic were of my own design.

In [ ]:
# all my imports, which include OCR stuff for converting manifesto pdfs to imgs for extraction, alongside other NLP libs
import numpy as np, pandas as pd
import sklearn
import logging
from pathlib import Path
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
import re
import unicodedata
from typing import List, Dict, Tuple, Optional
from collections import Counter
import json
import wordfreq
import nltk
import string
from nltk.tokenize import sent_tokenize
import os
import hashlib
from datetime import datetime

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
logging.getLogger('pdfminer').setLevel(logging.ERROR)

print("All libraries loaded in, ready to pull sentences out of the manifestos now.")

## Loading the Party Code Map

The MARPOR manifesto files I fetched only give a numeric party code, not a name. I therefore had to build this lookup table to translate between the two. I queried it from the [ManifestoR API](https://manifestoproject.wzb.eu/manifestor) and it's included in my submission in this same folder.

In [ ]:
# 1st, the party map is needed, as files are saved with a party code and date corresponding to a name and election date
map_of_uk_parties_to_marpor_codes= pd.read_csv("party_map_uk.csv", encoding="utf-8", usecols=["party", "partyname"])
print(f"map_of_uk_parties_to_marpor_codes is loaded in, there's ({len(map_of_uk_parties_to_marpor_codes)} parties total). Here's 3 samples from it:")
display(map_of_uk_parties_to_marpor_codes.sample(3))

## PDF Extraction + Artefact Repairing Functions

Most of the manifesto corpus is only available as custom-formatted PDFs with odd texts and layouts, which means I can't just read structured text in nicely from a txt or word doc. Instead I have to run OCR (which means optical character recognition) on each page and deal with the artefacts it introduces, of which there are quite a few.

The most common problem I ran into is this double-character run, where OCR duplicates each letter due to unusual font encodings (e.g. `tthhee` -> `the`). I correct this conservatively, with numerical tokens only being de-doubled in my code when they're adjacent to confirmed doubled words, as a value like `2244` could be valid on its own.

The other recurring one is character-level spacing, where individual letters get separated by whitespace (e.g. `t h e`). I detect this by checking what proportion of tokens are single characters. If it's high enough, a joining pass runs first before the double-character correction.

In [ ]:
# this function detects if a text string has characters separated by spaces
# I kept hitting this in older manifesto PDFs with weird font encodings
# e.g. "1 6 B u i l d i n g" would return True here
def check_if_sentence_contains_spaced_chars(text_for_spacedchar_test):
    words_in_sentence= text_for_spacedchar_test.split()

    # I exclude 'a' and 'I' because they're legitimate single-character words, but if there's a high proportion of other single chars it's a pretty strong signal of spaced out chars
    single_char_count= sum(1 for word in words_in_sentence if len(word) == 1 and word.lower() not in {'a', 'i'})

    # if more than 40% of tokens are single chars it's pretty much definitely one of those spaced out words from what I've seen
    if len(words_in_sentence) >0 and single_char_count / len(words_in_sentence) >0.4:
        return True

    # i also catch it via consecutive single-char tokens. 5 in a row is a clear giveaway for instance
    consec_single_chars, max_consecs_permitted= 0, 0
    for word in words_in_sentence:
        if len(word)== 1 and word.lower() not in {'a', 'i'}:
            consec_single_chars+= 1
            max_consecs_permitted= max(max_consecs_permitted, consec_single_chars)
        else:
            consec_single_chars= 0

    return max_consecs_permitted >= 5


# this rejoins chars that got separated by spaces from after i ran OCR
# e.g. "1 6 b u i l d i n g a n" -> "16 building an" is one example I found
def fix_spaced_out_chars(spaced_char_candidate_fix):
    if not check_if_sentence_contains_spaced_chars(spaced_char_candidate_fix):
        return spaced_char_candidate_fix
    words_in_sentence = spaced_char_candidate_fix.split()
    fixed_words, collector_of_accumulated_chars= [], []
    for word in words_in_sentence:
        # here i accumulate single-char tokens that aren't valid standalone words_in_sentence
        if len(word)== 1 and word.lower() not in {'a', 'i'}:
            collector_of_accumulated_chars.append(word)
        else:
            # I clean out the char collector accumulator when I hit a proper multi-char token though
            if collector_of_accumulated_chars:
                fixed_words.append(''.join(collector_of_accumulated_chars))
                collector_of_accumulated_chars= []
            fixed_words.append(word)

    # & clean out anything left at the end of it all too
    if collector_of_accumulated_chars:
        fixed_words.append(''.join(collector_of_accumulated_chars))

    return ' '.join(fixed_words)


In [ ]:
# this fixes double-char runs from weird font encodings in PDF extraction
# e.g. "FFoorr tthhee-> "For the"
def fix_double_char_run(sentence_to_fix):
    # spaced character correction runs first, as it's a prerequisite
    sentence_to_fix= fix_spaced_out_chars(sentence_to_fix)

    words_in_that_sentence= sentence_to_fix.split()
    words_fixed,points_where_doubles_exist= [], [check_for_if_word_is_doubled(word) for word in words_in_that_sentence]
    for i, word in enumerate(words_in_that_sentence):
        if points_where_doubles_exist[i]:
            words_fixed.append(''.join(word[wo] for wo in range(0, len(word), 2)))

        elif check_if_num_token(word) and check_if_num_is_doubled(word):
            # I only de-double a number if it's sitting next to a confirmed doubled word
            # otherwise something like "1122" could actually be a legitimate value
            check_for_doubled_neighbor= ((i > 0 and points_where_doubles_exist[i - 1]) or (i < len(words_in_that_sentence) - 1 and points_where_doubles_exist[i + 1]))
            if check_for_doubled_neighbor==True:
                words_fixed.append(''.join(word[wo] for wo in range(0, len(word), 2)))
            else:
                words_fixed.append(word)
        else:
            words_fixed.append(word)

    return ' '.join(words_fixed)


# here I consider a word doubled if >70% of its adjacent letter pairs are identical
def check_for_if_word_is_doubled(doublecheck_word):
    if len(doublecheck_word) < 4:
        return False
    indicies_of_adjacent_letters= [index_of_letter for index_of_letter, letter_itself in enumerate(doublecheck_word) if letter_itself.isalpha()] # only for alphanumerics though, nums wouldn't count
    if len(indicies_of_adjacent_letters) < 2:
        return False

    count_of_all_doubles, overall_pairs= 0, 0

    for index in range(0, len(indicies_of_adjacent_letters) -1, 2):
        first_index, second_index= indicies_of_adjacent_letters[index], indicies_of_adjacent_letters[index+1]
        if second_index== first_index +1 and doublecheck_word[first_index]== doublecheck_word[second_index]:
            count_of_all_doubles += 1
        overall_pairs += 1
    return overall_pairs >0 and (count_of_all_doubles / overall_pairs) >0.7


# the next two handle the tricky edge case of doubled numerical tokens
# "2244" on its own is ambiguous (24 could be a header count or 2244 the cnt of something), but pretty much always going to be an artefact when it appears alongside doubled words

# this returns True if at least half the token's chars are digits (so "1%" would count)
def check_if_num_token(word):
    cleaned_up_word= word.strip('.,!?;:') # I'm getting rid of rubbish to fairly analyse it
    if bool(cleaned_up_word) and sum(char.isdigit() for char in cleaned_up_word) / len(cleaned_up_word) >= 0.5:
        return True
    return False

# & returns True only if every digit pair is an exact repeat, e.g. 2244 yes, 2024 no
def check_if_num_is_doubled(num_dble_check_word):
    digits_present= [letter for letter in num_dble_check_word if letter.isdigit()]

    if len(digits_present) <2 or len(digits_present) % 2 != 0: # because can't be doubled num
        return False

    # but else it could be, so I run the check on it
    if all(digits_present[char_pos]== digits_present[char_pos + 1] for char_pos in range(0, len(digits_present), 2))== True:
        return True
    return False


# some tests for this, commented out for now as I've validated myself, but if you want to check then go ahead and uncomment + run
# fix_dble_run_txt= "2244 PPllaaiidd CCyymmrruu MMaanniiffeessttoo 22002244: This will allow us to run for government."
# test_non_dble_run_num= "The year 2244 could be great."
# print(fix_double_char_run(fix_dble_run_txt))
# print(fix_double_char_run(test_non_dble_run_num))


# Actual OCR Extraction
The above are validiation / preprocessing functions. From here on out I do full OCR extraction on all of the manifestos. In doing this I a) pull out all the relevant text then b) sort through it & discard anything that's corrupted / unreadable based on all my criteria I've defined in this notebook.

I did try pdfplubmer which doesn't use OCR and is much quicker but it annoyingly produced way more artefacts than this technique, so I stuck with Tesseract.

In [ ]:
# Now for my OCR-based PDF extraction, where I'm rendering each page as an img before running Tesseract on it.

# this main function returns the full extracted text as a single string, w/ pages separated by x2 newlines
def pullout_individual_manifesto_sentences_from_manifesto(pdf_path, dpi=300, lang='eng'): # 300dpi seems to produce accurate sentences and isn't crazy slow at least
    manifesto_pages_to_read= []

    try:
        print(f"Next up i'm converting manifesto {pdf_path.name} to imgs...")

        # each page here is getting rendered as a PNG img before OCR runs on it
        manifesto_page_imgs= convert_from_path(pdf_path, dpi=dpi, fmt='png', thread_count=2)#png keeps low size high quality
        print(f"now running OCR processing on {len(manifesto_page_imgs)} pages...")

        for page_num, manif_img in enumerate(manifesto_page_imgs, 1):
            # I chose PSM 3 as it has fully automatic page segmentation w/ no orientation detection, 
            # which should deal w/ the varied layouts of manifestos pretty well
            custom_setup_for_manifestos= r'--oem 3 --psm 3'
            page_text= pytesseract.image_to_string(manif_img, lang=lang, config=custom_setup_for_manifestos)

            if page_text.strip():  # I only add non-empty pages, saves a bit of space as there weirdly are some of these in manifestos
                manifesto_pages_to_read.append(page_text)

            if page_num % 10== 0: # print progress, but only every 10 otherwise it gets really visually busy in this cell
                print(f"Processed {page_num}/{len(manifesto_page_imgs)} pages")

        print(f"OCR extraction of that manifestos now done.")

    except Exception as e:
        print(f"Uh oh, there was a problem extracting the file: {pdf_path.name}: {e}")
        return ""

    combined_manifesto_text= "\n\n".join(manifesto_pages_to_read)
    return combined_manifesto_text


# NFKC normalisation, which I applied before anything else so the downstream regex patterns don't need to deal w/ composed vs decomposed unicode forms (like e and e w/ accent)
def normalise_unicode(manif_section):
    return unicodedata.normalize('NFKC', manif_section)


In [ ]:
# these are the repair pattern_in_the_manifs I've defined that need to be carried out. I've set it up this way for the log.
# I apply these in sequence; the log_label gets written to the repair log for traceability in my full version (see intro pargraph for that)
MANIF_REPAIRS_TO_MAKE= [
    # This fixes hyphenated line breaks (e.g. "environ-\nment"->"environment")
    (r'(\w+)-\s*\n\s*(\w+)', r'\1\2', 'hyphen_linebreak'),

    # this addresses some of the common micro-fusions (little errors with common terms) I found
    # I just used simple regex matching to find these instances & swap them out
    (r'\bina\b', 'in a', 'micro_fusion_ina'),
    (r'\bwil\b', 'will', 'micro_fusion_wil'),
    (r'\bforthe\b', 'for the', 'micro_fusion_forthe'),
    (r'\bandthe\b', 'and the', 'micro_fusion_andthe'),
    (r'\btothe\b', 'to the', 'micro_fusion_tothe'),
    (r'\binthe\b', 'in the', 'micro_fusion_inthe'),
    (r'\bofthe\b', 'of the', 'micro_fusion_ofthe'),
    (r'\bwil\s+be\b', 'will be', 'micro_fusion_wil_be'),

    # and this pushes multiple spaces into a single one
    (r'\s+', ' ', 'pushtogether_multiple_spaces'),
]


# these are all based off commonly known encoding artefact errors + specific ones i found, for removing all the weird char translation errors
MANIF_replacement_for_artefactS_TO_MAKE= {
    'â€™': "'", # for apostrophe
    'â€œ': '"', 'â€�': '"', # left & right quote
    'â€"': '–', # Em dash (uk party manifestos realistically would use nomral "-", I think em dash is an Americanism)
    'Â£': '£', 'â‚¬': '€', # (British) pound and then Euro sign
    'Â': '', # * I read that this one's common and has no translation, and did see it here and there
}


# now I have this, which applies document-level OCR repairs to the raw extracted text in order
# note: it returns (repaired_manif_text, manif_repair_log). The log I setup is for recording correction type, examples_for_fixup_manual_validating & counts
def fixup_text_from_manif(manif_text):
    manif_repair_log= []
    repaired_manif_text= manif_text
    repaired_manif_text= fix_double_char_run(repaired_manif_text)

    # I swap out the known encoding artefacts first
    for artefact_to_swap, replacement_for_artefact in MANIF_replacement_for_artefactS_TO_MAKE.items():
        repaired_manif_text= repaired_manif_text.replace(artefact_to_swap, replacement_for_artefact)

    for pattern_in_the_manif, replacement_for_artefact, pattern_in_the_manif_name in MANIF_REPAIRS_TO_MAKE:
        cumulative_manif_hits= list(re.finditer(pattern_in_the_manif, repaired_manif_text, re.IGNORECASE))
        if cumulative_manif_hits:
            # I keep a small sample of matched strings for manual validation
            examples_for_fixup_manual_validating= [repaired_manif_text[hit.start():hit.end()] for hit in cumulative_manif_hits[:5]]
            repaired_manif_text= re.sub(pattern_in_the_manif, replacement_for_artefact, repaired_manif_text, flags=re.IGNORECASE)

            manif_repair_log.append({
                'pattern_in_the_manif_name': pattern_in_the_manif_name,
                'original_examples_for_fixup_manual_validating': examples_for_fixup_manual_validating,
                'count': len(cumulative_manif_hits), 'replacement_for_artefact': replacement_for_artefact})

    return repaired_manif_text, manif_repair_log


# minimal pre-segmentation cleaning, which I'm deliberately not stripping punctuation or stopwords from because sent_tokenize needs them to find sentence boundaries reliably
def cleanup_manifest_text_for_segmenting(manif_text_for_cleanup):
    # NFKC normalisation first
    manif_text_for_cleanup= normalise_unicode(manif_text_for_cleanup)

    # strip null bytes and control characters, but keep newlines and tabs
    manif_text_for_cleanup= ''.join(char for char in manif_text_for_cleanup if char== '\n' or char== '\t' or not unicodedata.category(char).startswith('C'))

    # I remove bullets and directional symbols, as there's no lexical content there
    manif_text_for_cleanup= re.sub(r'[©•>→❱■¥●»♦]', '', manif_text_for_cleanup)

    # footnote references like [1], [23] etc can go
    manif_text_for_cleanup= re.sub(r'\[\d+\]', '', manif_text_for_cleanup)

    # I also fix gaps after capital letters left by bullet removal (e.g. "S upport" -> "Support")
    manif_text_for_cleanup= re.sub(r'\b([A-Z])\s+([a-z])', r'\1\2', manif_text_for_cleanup)

    # finally I normalise whitespace
    manif_text_for_cleanup= re.sub(r'[ \t]+', ' ', manif_text_for_cleanup)
    manif_text_for_cleanup= re.sub(r'\n{3,}', '\n\n', manif_text_for_cleanup)
    manif_text_for_cleanup= manif_text_for_cleanup.strip()

    return manif_text_for_cleanup


## Sentence Segmentation

Now I've setup all the repair functions I can now split the cleaned manifesto text into individual sentences. SBERT here is running sentence level & using sentence embeddings to explore the local lexical neighbourhood. Because of that I have to make sure I have proper sentence boundaries and not just token boundaries, which my functions beneath address.

In [ ]:
# identifies and reverses spaced-out capital letter sequences
# artefact I found in some of the older PDFs where individual characters get space-separated
# and also end up in the wrong order
def repair_kerning_in_manif(sentence):
    instance_of_kerning_pattern= r"(([A-Z]\s(?=[A-Z]\s|[A-Z]$)){2,}[A-Z])" # after a while of experimentation, this is the pattern that best captures the kerning i'm seeing in these manifestos
    kerning_repair_logs= [] # and as usual, this is logged when a kerning repair is made
    def process_match(match):
        original_kerned_err= match.group(0)
        fixed_dekerned_str= original_kerned_err.replace(" ", "")[::-1]

        # i log it so I can see what got fixed
        kerning_repair_logs.append({
            'type': 'kerning_reversal',
            'original_kerned_err': original_kerned_err, 'fixed_dekerned_str': fixed_dekerned_str,
            'count': 1})
        return fixed_dekerned_str

    # re.sub accepts a function as the replacement argument, which is handy here for my helper function
    corrected_sentence= re.sub(instance_of_kerning_pattern, process_match, sentence)
    return corrected_sentence, kerning_repair_logs

# quick test using one of the kerned sentneces i found, checking it reverses the spaced caps back correctly
test_str= "FOR THE D O O G  N O M M O C GENERAL ELECTION MANIFESTO 2015"
print(repair_kerning_in_manif(test_str))
# should give: FOR THE COMMON GOOD GENERAL ELECTION MANIFESTO 2015


In [ ]:
big_list_of_english_words= set(wordfreq.top_n_list("en", n=100000))
# 1st & foremost it requires 2+ character tokens to avoid single-letter noise
def english_coverage(text, big_en_vocab):
    manif_sent_toks= re.findall(r'\b[a-z]{2,}\b', text.lower())
    if not manif_sent_toks: return 0.0
    return sum(m_t in big_en_vocab for m_t in manif_sent_toks) / len(manif_sent_toks)

In [ ]:
# Irish manifesto files contain the full content written twice; 1x in English, 1x in Gaelic. I just chose to use an English vocab coverage to drop the Gaelic sentences.
# I set a minimum character count before judging, so short sentences don't get incorrectly filtered
def filterout_non_english_sentences(sentence_to_check_if_eng, most_prevelant_eng_vocab, min_coverage= 0.6, min_chars_present= 4):
    nonenglish_sentences_cleanedup_logs= [] # business as usual w/ logs

    chars_present= re.findall(r'\b[a-z]{2,}\b', sentence_to_check_if_eng.lower())
    if len(chars_present) < min_chars_present:
        return sentence_to_check_if_eng, nonenglish_sentences_cleanedup_logs  # too short to judge reliably, so I keep it

    coverage_eng_test= english_coverage(sentence_to_check_if_eng, most_prevelant_eng_vocab)
    if coverage_eng_test < min_coverage:
        nonenglish_sentences_cleanedup_logs.append({
            "type": "non_english_sentence_dropped", "english_coverage": round(coverage_eng_test, 3), # (as %)
            "original_preview": sentence_to_check_if_eng[:120]})
        return "", nonenglish_sentences_cleanedup_logs

    return sentence_to_check_if_eng, nonenglish_sentences_cleanedup_logs


## now the test to see if this works as usual, using a set of valid/invalid/partially valid sentences
# testing_english_filter_works_sentences= {
#     "english_clear": "We believe in balancing the budget in a fair and responsible way.",
#     "irish_clear": "Creidimid i gcothromaíocht an bhuiséid ar bhealach cóir agus freagrach.",
#     "irish_clear_2": "Tá an páirtí seo tiomanta do cheartas sóisialta agus comhionannas.",
#     "mixed_short": "Tá equality important.",
#     "english_short": "This is fine.",
# }
# print("now running a test to see if the filter works...")

# for lang_state_lbl, sentence_to_test_if_eng in testing_english_filter_works_sentences.items():
#     filtered_out, nonenglish_sentences_cleanedup_logs= filterout_non_english_sentences(sentence_to_test_if_eng, most_prevelant_eng_vocab, min_coverage=0.6)
#     coverage_eng_test= english_coverage(sentence_to_test_if_eng, most_prevelant_eng_vocab)
#     print(f"test case is:{lang_state_lbl}")
#     print(f"original sentence:{sentence_to_test_if_eng}")
#     print(f"english coverage:{coverage_eng_test:.3f}")
#     print(f"and output: {filtered_out if filtered_out else '[DROPPED!!]'}")
#     print(f"nonenglish_sentences_cleanedup_logs: {nonenglish_sentences_cleanedup_logs}")


In [ ]:
# this cell's job is to strip garbage numerical prefixes from sentence starts. I take care here to avoid stripping meaningful ones though
# 2 examples would be: "9.7 Tackling terrorism..." -> "Tackling terrorism..." and "8 - - 10 10 11 1 7 Northern Ireland" -> "Northern Ireland"
# but 3+ digit numbers, %s and nums followed by units/year ranges get left alone, as those could and probably are valid in a manifesto context


# to do this, I define a regex pattern the encapsulates what I see as "safe" numbers / letters, and filterout whatever doesn't match that but still meets the preconditions for removal
VALID_NUMERICAL_MANIF_PATTERN= r'(?:%|per cent|percent|pounds|£|millions|million|billions|billion|thousand|thousands)'
VALID_YR_SUFFIC_MANIF_PATTERN= r'(?:/\d{2,4})'
DETECTOR_OF_SAFE_MANIF_NUM_PATTERNS= re.compile(r'^(?:\d{3,}|%|\d{1,2}\s*(?:' + VALID_NUMERICAL_MANIF_PATTERN + r'|' + VALID_YR_SUFFIC_MANIF_PATTERN + r'))', re.IGNORECASE)

# so now this removes garbage numerical/punctuation prefixes from sentences that start with 1-2 digits while protecting numbers that are actually meaningful (e.g. "2% of GDP", "2024 Manifesto")
def fixup_leading_nums_from_manif_sentence_start(manif_sentence):
    # quick check: does it even start with a digit? if no, no need to filter anything
    if not re.match(r'^\d', manif_sentence):
        return manif_sentence, []

    # if it matches the meaningful number pattern I defined above, I leave it alone
    if DETECTOR_OF_SAFE_MANIF_NUM_PATTERNS.match(manif_sentence):
        return manif_sentence, []

    # otherwise I strip the prefix, matching anything that's digits/punctuation/spaces up to the first letter, then remove it
    dodgy_numeral_match_found= re.match(r'^([\d\W_]+)(?=[a-zA-Z])', manif_sentence)

    # so that gets sorted here
    if dodgy_numeral_match_found:
        garbage_prefix= dodgy_numeral_match_found.group(1)
        cleanedup_sentence= manif_sentence[len(garbage_prefix):].strip()
        if not cleanedup_sentence:
            return manif_sentence, []
        return cleanedup_sentence, [{'type': 'leading_numeral_strip', 'original': manif_sentence[:50], 'removed': garbage_prefix, 'count': 1}]

    return manif_sentence, []


The repair log I talk about below is something I decided to build in because there's so many repairs going on here (literally thousdands per manifesto sometimes) that I thought it ought to be logged for reproducibility's sake.

### Filtering Document Metadata and Structural Artefacts

The raw OCR output I fetched is full of a lot of structural noise like section header numbers, repeated running titles, table-of-contents entries etc. None of those actually carry any meaningful semantic content for my project's analysis, so I'm just removing them. I do that using a mix of pattern-matching and repetition detection.

In [ ]:
# checks whether any numbers in a sentence appear in a meaningful content context
# (as opposed to just being a page number or section header)
def has_meaningful_number_context(sentence):
    meaningful_patterns_manif_could_have= [
        r'\d+\s*%',  # %s
        r'\d+\s*per\s*cent',
        r'\d{3,}',# 3+ digits
        r'\d+\s*(?:billion|million|thousand|pounds|£|bn|jobs|people)',
        r'\d{4}/\d{2,4}', # yr ranges
    ]
    return any(re.search(meaningful_manif_pattern, sentence, re.IGNORECASE) for meaningful_manif_pattern in meaningful_patterns_manif_could_have)


# a set of heuristics for detecting whether a sentence is actually a header
def is_likely_header(sentence):
    if len(sentence) < 50 and re.match(r'^\d{1,2}\s+', sentence):
        the_rest= re.sub(r'^\d+\s*', '', sentence)
        manif_alpha_chars= [m_ch for m_ch in the_rest if m_ch.isalpha()]
        if manif_alpha_chars and sum(num_manif_char.isupper() for num_manif_char in manif_alpha_chars)/len(manif_alpha_chars) > 0.5:
            return True
    # all these are just searches for typical header stuff you'd often read in manifestos, guided by my failed extraction attempts
    if re.search(r'(?:Layout|Page|v)\s+\d+', sentence, re.IGNORECASE):
        return True
    if re.search(r'^\d+\s+\d+\s+\d+', sentence):
        return True
    if re.match(r'^\d{1,2}\s+[A-Z\s]+$', sentence) and len(sentence) < 40:
        return True
    # I got (cid:XX)'s a lot,, which is a pretty definite sign of a font-encoding artefact & not real content
    if re.search(r'\(cid:\d+\)', sentence):
        return True

    cleanedup_sent= re.sub(r'^\d+\s*', '', sentence).strip()

    # 25+ chars of all-caps at the start strongly suggests a heading
    if re.match(r'^[A-Z\s]{25,}', cleanedup_sent):
        # but not if the whole thing is caps, as that might actually be valid emphasis
        if not cleanedup_sent.isupper():
            return True
    # but chapter markers are reliably non-content
    if re.search(r'CHAPTER\s+\d+', sentence, re.IGNORECASE):
        return True
    
    return False


# this checks if this sentence appears 3+ times across the corpus, which is a strong sign it's a repeated header
def is_repeated_header(sentence, big_collection_of_manif_sents):
    core_manif_sent_text= re.sub(r'^\d+\s*', '', sentence).strip()
    if len(core_manif_sent_text) < 10:
        return False
    similar_count= sum(1 for mn_sent in big_collection_of_manif_sents if re.sub(r'^\d+\s*', '', mn_sent).strip()==core_manif_sent_text)
    return similar_count >= 3


# extracts the repeated or all-caps pattern from a sentence that I use as the matching key handles both internal duplicates
# ("Caring For Everyone Caring For Everyone") and all-caps headers ("12 FOR THE MANY NOT THE FEW content...")
def extract_repeated_pattern(sentence):
    cleaned= re.sub(r'^\d+\s*', '', sentence).strip()
    # all-caps at the start is the most common header pattern I've seen
    # match 10+ consecutive capital letters/spaces until lowercase or end of string
    all_caps_match= re.match(r'^([A-Z\s]{10,}?)(?=[a-z]|$)', cleaned)
    if all_caps_match:
        caps_pattern= all_caps_match.group(1).strip()
        # needs at least 2 words to be meaningful
        if len(caps_pattern.split()) >= 2:
            return caps_pattern

    # next, check for internally repeated phrases
    candidate_manif_words= cleaned.split()

    for length in range(2, min(15, len(candidate_manif_words) //2 + 1)):
        pattern= ' '.join(candidate_manif_words[:length])
        pattern_escaped= re.escape(pattern)
        matches= list(re.finditer(r'\b' + pattern_escaped + r'\b', cleaned, re.IGNORECASE))
        if len(matches) >= 2:
            return pattern

    # if the sentence is short and mostly caps, just return the whole thing
    if len(cleaned) <50:
        alpha_manif_chars= [mn_c for mn_c in cleaned if mn_c.isalpha()]
        if alpha_manif_chars and sum(c.isupper() for c in alpha_manif_chars) / len(alpha_manif_chars) > 0.7:
            return cleaned
    return cleaned


# this normalises a header pattern for fuzzy comparison (lowercases it, strips its nums & collapses its whitespace)
def normalise_header_pattern(pattern):
    normalised_pat= pattern.lower()
    normalised_pat= re.sub(r'\d+', '', normalised_pat)
    # (i'm leaving the cid removal commented out as it actually didn't help in practice)
    # normalised_pat= re.sub(r'\(cid:\d+\)', '', normalised_pat)
    return normalised_pat


# I setup this function to check if a sentence contains a pattern that appears repeatedly across the corpus
def is_fuzzy_repeated_header(sentence, all_sentences, min_occurrences=3):
    manif_pattern= extract_repeated_pattern(sentence)

    # I lowered the minimum from 10 to 8 characters to catch shorter patterns
    if len(manif_pattern) <8:
        return False, None, 0

    normalised_manif_pattern= normalise_header_pattern(manif_pattern)

    if len(normalised_manif_pattern) <5:
        return False, None, 0

    manif_occurrence_count= 0
    for other_sentence in all_sentences:
        other_pattern= extract_repeated_pattern(other_sentence)
        other_normalised= normalise_header_pattern(other_pattern)

        if (other_normalised == normalised_manif_pattern or normalised_manif_pattern in other_normalised or other_normalised in normalised_manif_pattern):
            manif_occurrence_count += 1

    is_header= manif_occurrence_count >= min_occurrences
    return is_header, manif_pattern if is_header else None, manif_occurrence_count


# this decides whether to filter a number-prefixed sentence & returns metadata for the log
def should_filter_number_sentence_enhanced(sentence, all_sentences=None):
    # i keep if it doesn't start with a digit
    if not re.match(r'^\d', sentence):
        return False, None

    # & also keep if the number is in a meaningful context
    if has_meaningful_number_context(sentence):
        return False, None

    if is_likely_header(sentence):
        return True, {'operation': 'sentence_filter', 'filter_type': 'likely_header_pattern', 'sentence_preview': sentence[:100]} # (a few sentences are massive so I cut them off to 100 chars)

    if all_sentences and is_repeated_header(sentence, all_sentences):
        core= re.sub(r'^\d+\s*', '', sentence).strip()
        return True, {'operation': 'sentence_filter', 'filter_type': 'exact_repeated_header','pattern': core[:50], 'sentence_preview': sentence[:100]}

    if all_sentences:
        is_fuzzy, pattern, count = is_fuzzy_repeated_header(sentence, all_sentences)
        if is_fuzzy:
            return True, {'operation': 'sentence_filter', 'filter_type': 'fuzzy_repeated_header', 'pattern': pattern[:50], 'occurrences': count,'sentence_preview': sentence[:100]}

    return False, None


# filters sentences that contain publication metadata artefacts (ISBNs, URLs, emails etc.)
def should_filter_metadata(sentence):
    # "about this publication" sections are always boilerplate
    if "about this publication" in sentence.lower():
        return True, {
            'operation': 'sentence_filter',
            'filter_type': 'publication_metadata',
            'match': 'about this publication'
        }

    # isbns are often present because manifestos are published docs but they only add noise to my downstream analysis
    isbn_pattern = r'\bISBN(?:[:=\-\s]+)(?:97[89][-\s]?)?(?:\d[-\s]?){9,}[\dxX]\b'
    if re.search(isbn_pattern, sentence, re.IGNORECASE):
        return True, {
            'operation': 'sentence_filter',
            'filter_type': 'isbn_metadata',
            'sentence_preview': sentence[:50]
        }

    # urls are noisy for my use too
    url_pattern = r'(?:https?://|www\.)\S+\b|(?:\bURL|\bWebsite)\s*[:=]\s*\S+\.\S+'
    if re.search(url_pattern, sentence, re.IGNORECASE):
        return True, {
            'operation': 'sentence_filter',
            'filter_type': 'url_metadata',
            'sentence_preview': sentence[:50]
        }

    # & emails...
    email_pattern = r'\b[\w\.-]+@[\w\.-]+\.\w{2,}\b'
    if re.search(email_pattern, sentence):
        return True, {
            'operation': 'sentence_filter',
            'filter_type': 'email_metadata',
            'sentence_preview': sentence[:50]
        }

    # & phones. This matches "Tel/Phone: +44..." or "020..." style numbers
    phone_pattern = r'(?:Tel|Phone|Mob)(?:ile)?\s*[:.]?\s*(?:\+?44|0)[\d\s-]{9,}'
    if re.search(phone_pattern, sentence, re.IGNORECASE):
        return True, {
            'operation': 'sentence_filter',
            'filter_type': 'phone_metadata',
            'sentence_preview': sentence[:50]
        }

    # this deals w/ the @handle, using 3+ chars to avoid noise like "@ 5", and not preceded by word char (avoids emails)
    social_pattern= r'(?<![\w])@\w{3,}\b'
    if re.search(social_pattern, sentence):
        return True, {'operation': 'sentence_filter', 'filter_type': 'social_media_handle', 'sentence_preview': sentence[:50]}

    postcode_pattern= r'\b[A-Z]{1,2}[0-9][A-Z0-9]?\s*[0-9][A-Z]{2}\b'
    imprint_pattern= r'\b(?:Promoted|Printed|Published)\s+by\s+[A-Z]'

    if re.search(postcode_pattern, sentence) or re.search(imprint_pattern, sentence, re.IGNORECASE):
        return True, {'operation': 'sentence_filter', 'filter_type': 'address_imprint', 'sentence_preview': sentence[:50]}

    return False, None


# trying the basic spaced-character case first, as I kept seeing this in older manifesto PDFs
test1 = "1 6 b u i l d i n g a n e c o n o m y t h a t w o r k s"
print("Spaced test:")
print("  Before:", test1)
print("  After:", fix_spaced_out_chars(test1))
print()

# testing header detection on the Labour 2017 "FOR THE MANY NOT THE FEW" headers, which were the main problem case I built this to handle
test_headers= [
    "12 FOR THE MANY NOT THE FEW UPGRADING (cid:40)ast.",
    "16 FOR THE MANY NOT THE FEW A NEW DEAL FOR BUSINESS The majority...",
    "18 FOR THE MANY NOT THE FEW WIDENING parcel charges...",
    "22 FOR THE MANY NOT THE FEW CHAPTER 02 NEGOTIATING BREXIT",
]

print("Header detection test:")
for i, header in enumerate(test_headers, 1):
    pattern = extract_repeated_pattern(header)
    normalized = normalise_header_pattern(pattern)
    is_fuzzy, pat, count = is_fuzzy_repeated_header(header, test_headers)

    print(f"\n{i}. Sentence: {header[:50]}...")
    print(f"   Pattern: '{pattern}'")
    print(f"   Normalized: '{normalized}'")
    print(f"   Is header: {is_fuzzy}, Count: {count}")


# and a quick check on the metadata filter against cases I'd actually seen in the PDFs
test_sentences= [
    "This is a valid policy sentence about the economy.",
    "ISBN: 978-3-16-148410-0",
    "For more information visit www.labour.org.uk today.",
    "Contact us at info@conservatives.com for details.",
    "Promoted by A. Smith at SW1A 0AA.",
    "Follow us @greenpartyuk on Twitter.",
    "Tel: 020 7222 9000 for support.",
    "About this publication: This document was created in 2024.",
    "URL: http://example.com"
]

print(f"{'fake manifesto test sentence':<50} | {'Action code takes':<10} | {'Reason for that decision'}")
for sent in test_sentences:
    should_filter, metadata= should_filter_metadata(sent)
    action_code_takes= "FILTER OUT!!" if should_filter else "keep, all good :)"
    reason_for_filter_decision= metadata['filter_type'] if metadata else "all good!"

    # this is a formatting trick I learnt that displays the examples more tabularly
    display_sent= (sent[:47] + '...') if len(sent) > 47 else sent
    print(f"{display_sent:<50} | {action_code_takes:<10} | {reason_for_filter_decision}")


### Correcting Reversed Text Sequences

Some of the PDFs produce a really annoying OCR failure where entire sentence would be pulled out in reverse character order. An example of that is the sentence `yaw riaf a ni tegdub eht gnicnalab | secnanfi elbisnopseR`, which is actually just a back-to-front version of *Responsible finances | balancing the budget in a fair way*.

I use two different sweeps of the sentence to hand that. First I do a full chara-level reversal where to x2 check the weird looking string is actually a valid sentence when it's backwards. Second, if on top of that the token is also back-to-front I do a reversal on that which recovers the grammatically correct version of it. I check both passes against an English vocab dict so I'm not accidentally reversing text that's already fine.

In [ ]:
# I load the English vocab once here and reuse it across all the reversal functions
REGEX_PATTERN_SPANNING_DELIMTERS= r"(\s*\|\s*|\s*:\s*|\s*–\s*)"
STOPWORDS= {"the","a","an","in","on","of","to","for","with","by","and","or","but"}

# i give a score to each stopword position for helping w/ this filtering of garbled and irish words
def stopword_position_score(sentence):
    manif_sent_toks= sentence.lower().split()
    if not manif_sent_toks: return 0.0
    positions = [i / len(manif_sent_toks) for i, m_t in enumerate(manif_sent_toks) if m_t in STOPWORDS]
    if not positions: return 0.0
    return 1.0 -sum(positions) / len(positions)

# i skip if well-formed; otherwise i try reversing the whole string
def fix_fully_reversed_sentence(sentence, vocab, min_len=5):
    check_full_reverse= english_coverage(sentence, vocab)
    if len(sentence) < min_len or check_full_reverse> 0.25: return sentence, []

    rev_text= sentence[::-1]
    # threshold of 0.45 seem enough from my experiments to catch partially misspelled reversed strings
    if english_coverage(rev_text, vocab) >0.45:
        return rev_text, [{'type': 'full_reverse_fix', 'original': sentence, 'repaired': rev_text}]
    return sentence, []

# this is quite a heavy function and took a fair bit of engineering, but it's designed as a 2nd stage patch once the reversal has been identified as being tokenwise
def fix_tokenwise_reversal(sentence, vocab):
    manif_sent_toks= sentence.split()
    first_sent= english_coverage(sentence, vocab)

    # short, well-formed text gets left alone
    if not manif_sent_toks or (len(manif_sent_toks) < 4 and first_sent> 0.3): return sentence, []

    rev_tokens= [m_t[::-1] for m_t in manif_sent_toks]
    charrev2= " ".join(rev_tokens)             # character reversal only
    charrev3= " ".join(reversed(rev_tokens))   # character reversal + token order reversal

    second_sent, third_sent= english_coverage(charrev2, vocab), english_coverage(charrev3, vocab)
    if second_sent<= first_sent and third_sent <= first_sent: return sentence, []  # neither candidate is better

    # I prefer the token-order variant (c3) via stopword position score + a small tie-breaker
    tok_o2, tok_o3= stopword_position_score(charrev2), stopword_position_score(charrev3)
    score2= second_sent+ 0.15 * tok_o2
    score3= third_sent + 0.15 * tok_o3 + 0.01
    best_sent= charrev3 if score3 > score2 else charrev2
    best_var= "char_plus_order" if score3 > score2 else "char_only"
    if (max(second_sent, third_sent) - first_sent) >= 0.4:
        return best_sent, [{'type': 'tokenwise_reverse_fix', 'variant': best_var, 'original': sentence, 'repaired': best_sent}]
    return sentence, []

# sometimes I found just the prefix of a manifesto sentence needs reversing
def fix_reversed_prefix(sentence, vocab):
    stripped_manif_sent= sentence.strip()
    if english_coverage(stripped_manif_sent, vocab) > 0.8: return sentence, []  # i leave well-formed text alone, no need to mess w/ it

    candidates_for_reversal_fixing= []
    # I'm testing prefix lengths from 15 to 120 chars
    for cut_into_manif_sent in range(15, min(120, len(stripped_manif_sent) - 10)):
        # I need to avoid cutting inside a word so I account for that here
        if cut_into_manif_sent> 0 and stripped_manif_sent[cut_into_manif_sent-1].isalnum() and stripped_manif_sent[cut_into_manif_sent].isalnum(): continue

        prefix, suffix= stripped_manif_sent[:cut_into_manif_sent], stripped_manif_sent[cut_into_manif_sent:]
        if not suffix.strip(): continue

        rev_cov= english_coverage(prefix[::-1], vocab)
        if english_coverage(prefix, vocab) < 0.25 and rev_cov > 0.6 and english_coverage(suffix, vocab) > 0.6:
            candidates_for_reversal_fixing.append({'cut': cut_into_manif_sent, 'cov': rev_cov, 'res': prefix[::-1].strip() + " " + suffix.lstrip(), 'orig': prefix})

    if candidates_for_reversal_fixing:
        # pick the best candidate by coverage quality, then prefix length
        best_rev_candidates_here= max(candidates_for_reversal_fixing, key=lambda x: (x['cov'], x['cut']))
        return best_rev_candidates_here['res'], [{'type': 'reversed_prefix_fix', 'original': best_rev_candidates_here['orig'], 'repaired': best_rev_candidates_here['res'][:50]}]
    return sentence, []

# and this applies sentence reversal fixes on a high lvl
def fix_sentence_reversals(sentence):
    sent_rev_fix_logs= []
    spans= re.split(REGEX_PATTERN_SPANNING_DELIMTERS, sentence)
    repaired_spans= [] # keeping track for logs purpose

    for span in spans:
        if re.fullmatch(REGEX_PATTERN_SPANNING_DELIMTERS, span or ""):
            repaired_spans.append(span)
            continue

        # I try full reversal first just because it's cheap and catches strings like "secnanif elbisnopseR"
        span, result_of_fixed_sent= fix_fully_reversed_sentence(span, big_list_of_english_words)
        if result_of_fixed_sent:
            sent_rev_fix_logs.extend(result_of_fixed_sent)
            repaired_spans.append(span)
            continue

        # then I go to token-wise reversal, which is more expensive but catches partial cases like "yaw riaf"
        span, result_of_fixed_sent= fix_tokenwise_reversal(span, big_list_of_english_words)
        sent_rev_fix_logs.extend(result_of_fixed_sent)
        repaired_spans.append(span)

    sentence= "".join(repaired_spans)

    # reversed prefix pass catches things like "5102 otsefinaM tarcomeD larebiL"
    sentence, result_of_fixed_sent = fix_reversed_prefix(sentence, big_list_of_english_words)
    sent_rev_fix_logs.extend(result_of_fixed_sent)

    return sentence, sent_rev_fix_logs


# now I test as per usual
# tests = [
#     "yaw riaf a ni tegdub eht gnicnalab",
#     "yaw riaf a ni tegdub eht gnicnalab | secnanif elbisnopseR",
#     "5102 otsefinaM tarcomeD larebiL For further information",
#     "This sentence should not change at all"
# ]

# for test in tests:
#     fixed, _, logs = fix_sentence_reversals(test)
#     print(f"\n original is: {test}")
#     print(f"& then repaired is: {fixed}")


In [ ]:
# splits cleaned manifesto text into individual sentences
def splitup_manif_into_sentences(manif_to_split):
    sentences_from_manif= sent_tokenize(manif_to_split, language='english')
    sentences_from_manif= [manif_sentence.strip() for manif_sentence in sentences_from_manif if manif_sentence.strip()]
    return sentences_from_manif


# this runs all sentence-level repairs in sequence & builds up a unified repair log
def apply_sentence_level_repairs(manif_sentence):
    repair_log= []

    # 1. filter non-English sentences first (Gaelic replication in Irish manifestos)
    manif_sentence, logs = filterout_non_english_sentences(manif_sentence, big_list_of_english_words)
    repair_log.extend(logs)
    if not manif_sentence.strip():
        return manif_sentence, len(repair_log), repair_log

    # 2. fix character-level and token-order reversals
    manif_sentence, rev_logs = fix_sentence_reversals(manif_sentence)
    for log in rev_logs:
        log['count'] = 1
    repair_log.extend(rev_logs)

    # 3. correct single-character kerning artefacts
    manif_sentence, kerning_logs = repair_kerning_in_manif(manif_sentence)
    repair_log.extend(kerning_logs)

    # 4. strip leading numerical garbage (page numbers, section prefixes etc.)
    manif_sentence, num_logs = fixup_leading_nums_from_manif_sentence_start(manif_sentence)
    repair_log.extend(num_logs)

    # 5. remaining PDF-level text repairs
    manif_sentence, pdf_logs= fixup_text_from_manif(manif_sentence)
    repair_log.extend(pdf_logs)

    overall_cnt_of_manif_repairs= sum(repair_i_made.get('count', 1) for repair_i_made in repair_log)
    return manif_sentence, overall_cnt_of_manif_repairs, repair_log


# filters out sentences that are too short, too long, mostly numeric,
# or match known table-of-contents / header artefact patterns.
# I also catch OCR garbage and repeated-character artefacts here,
# as these slip through the earlier repair stages
def filter_sentences(sentences, min_length=10, max_length=1500):
    filtered= []

    for sent in sentences:
        sent_len = len(sent)

        # these are length bounds, so "Is a." gets caught here
        if sent_len < min_length or sent_len > max_length:
            continue

        # skip if predominantly numeric, as that's probably a table
        if sum(c.isdigit() for c in sent) / len(sent) > 0.5:
            continue

        # no alphabetic content at all is likely either a table or TOC too
        if not any(c.isalpha() for c in sent):
            continue

        # too many dots = TOC leader line (e.g. "Britain ..............")
        if sent.count('.') / len(sent) > 0.2:
            continue

        # page header artefacts like "Britain Together | 61"
        if re.search(r'\|\s*\d+\s*[A-Z]', sent):
            continue

        # skip if too many non-alphanumeric characters. this catches OCR artefacts like "jd LFr=J|4r hs Fa" that pass the other checks
        alnum_count = sum(c.isalnum() or c.isspace() for c in sent)
        if alnum_count / max(sent_len, 1) < 0.7:
            continue

        # skip if dominated by repeated character runs, as catches corrupted TOC reads like "cecceeccccesssccsssecsese..." and OCR stutter
        # patterns like "AAAMAAAS eed LAAAAAS"
        if re.search(r'(.)\1{4,}', sent):
            continue

        # skip if fewer than 40% of the 2+ char tokens are real English words. I'm using the same vocab loaded for the reversal functions.
        # this catches both OCR garbage (which has ~0-10% coverage) and any remaining Gaelic sentences (~10-30% coverage) that the dedicated
        # non-English filter missed due to short token counts
        eng_tokens_in_this_manif= re.findall(r'\b[a-z]{2,}\b', sent.lower())
        if len(eng_tokens_in_this_manif) >= 3:
            eng_ratio = sum(word in big_list_of_english_words for word in eng_tokens_in_this_manif) / len(eng_tokens_in_this_manif)
            if eng_ratio < 0.4: # granted this is arbritary but it seems to strike a good balance from my experimentations
                continue
        filtered.append(sent)
    return filtered


print("Sentence segmentation all functions locked and loaded for running on corpus!!")

### Selective Capitalisation Removal

When analysing what came out of the extraction I noticed that all-caps sentences almost always come from document-level formatting like section headings, chapter markers, TOC entries etc that got merged with a part of a sentence that otherwise would've been perfectly valid. Instead of having to strip away all capitalisation I'm removing it carefull w/ the conditions:

 a) runs of three or more consecutive all-caps words get stripped 

 b) anything following a `CHAPTER` marker is removed through to the next lowercase token 
 
 c) any punctuation or numbers left over from those removals get cleaned up too

In [ ]:
# removes all-caps formatting runs under the three conditions described in the cell above
def remove_selective_capitals(text):
    # here I define all the regex stuff needed that's specific to manifesto formatting
    NUMERICAL_UNIT_PATTERN= r'(?:%|per cent|percent|pounds|£|millions|million|billions|billion|thousand|thousands)'
    YEAR_SUFFIX_PATTERN= r'(?:/\d{2,4})'
    SAFE_NUMBER_GROUP_REGEX= re.compile(r'^(?:\d{3,}|%|\d{1,2}\s*(?:' + NUMERICAL_UNIT_PATTERN + r'|' + YEAR_SUFFIX_PATTERN + r'))',re.IGNORECASE)
    
    # I strip everything from "CHAPTER" onwards through to the next lowercase token
    chapter_pattern = r'\b[0-9\s]*(?:CHAPTER|Chapter)\s*[0-9]*[\s:;,.\-]*[A-Z\s]+(?:[:\-;,.].*?)(?=\b[a-z]|$)'
    text= re.sub(chapter_pattern, ' ', text)

    # here I remove runs of 3+ consecutive all-caps words
    caps_sequence_pattern= r'\b(?:[A-Z]{2,}(?:\s+[A-Z]{2,}){2,})\b'
    text= re.sub(caps_sequence_pattern, ' ', text)

    # clean up any orphaned numbers/punctuation at the start of the string but only if SAFE_NUMBER_GROUP_REGEX doesn't match (so meaningful numbers are left alone)
    if not SAFE_NUMBER_GROUP_REGEX.match(text.strip()):
        text= re.sub(r'^\s*[0-9:;,.\-]+\s*', '', text)

    # i always clean up trailing numerical artefacts (usually page numbers)
    text= re.sub(r'\s+[0-9:;,.\-]+\s*$', '', text)

    # + normalise any resulting multiple spaces
    text= re.sub(r'\s+', ' ', text).strip()

    return text


# # my test cases again, where I'm just checking the main cases work as expected
# test_cases = [
#     "CHAPTER 2: SOMETHING; today we discuss policy.",
#     "INTRODUCTION This is the main text.",
#     "We need A NEW VISION FOR BRITAIN anti-drug content in our manifesto.",
#     "I am happy that the SECTION A protocol was followed.",
#     "Some text CHAPTER 1 THE ECONOMY: GROWTH AND STABILITY and then some more text here.",
#     "The ECONOMY is important.",
# ]

# for test in test_cases:
#     cleaned = remove_selective_capitals(test)
#     print(f"Original: {test}")
#     print(f"Cleaned:  {cleaned}")
#     print()


## Main Processing Pipeline

With all my component functions for repairing now defined, this is where everything comes together. I take each manifesto from raw text through OCR repair, cleaning, segmentation, metadata filtering & sentence-level correction in sequence, and then write all operations to a unified log as I go.

In [ ]:
# I use this function to processes a single manifesto from raw text to sentence-level records,
# returning a unified operation log that covers all repairs, filters, and transformations
def process_single_manifesto(party_code, election_year, name_of_manif_file, raw_text, party_name=None):
    manifesto_id= f"{party_code}_{election_year}" # this is the same id I use to store the manifesto pdfs in my project repo

    # single log for everything that makes downstream analysis much easier
    running_log_of_manif_repairs= []

    if not raw_text:
        print(f"uh oh, problem found; {manifesto_id} has no text to work with here.")
        return [], []

    # step 1 I document-level PDF repair
    repaired_text, document_repair_log= fixup_text_from_manif(raw_text)
    for repair in document_repair_log:
        repair['operation'] = 'pdf_repair'
        repair['manifesto_id'] = manifesto_id
        repair['filename'] = name_of_manif_file
        running_log_of_manif_repairs.append(repair)

    # step 2 is minimal pre-segmentation cleaning
    cleaned_text= cleanup_manifest_text_for_segmenting(repaired_text)

    # step 3 is then sentence segmentation
    sentences= splitup_manif_into_sentences(cleaned_text)
    original_manif_sent_cnt= len(sentences)

    # & step 4 is filtering out garbage sentences (too short, too numeric etc.)
    sentences= filter_sentences(sentences)
    garbage_count= original_manif_sent_cnt - len(sentences)
    if garbage_count > 0:
        running_log_of_manif_repairs.append({
            'operation': 'garbage_filter', 'manifesto_id': manifesto_id, 'filename': name_of_manif_file, 'sentences_filtered': garbage_count,
            'filter_types': 'length, mostly_numbers, no_alpha, too_many_dots, page_headers'})

    # + also includes selective capitalisation removal
    sentences = [remove_selective_capitals(s) for s in sentences]

    # & filtering number-based headers and metadata artefacts w/ the detailed log included
    sentences_pre_filter= sentences.copy()
    sentences_i_kept= []
    filtered_by_reason= Counter()

    for sentence in sentences_pre_filter:
        # check 1: is it a header or numerical garbage?
        should_filter, filter_metadata = should_filter_number_sentence_enhanced(sentence, sentences_pre_filter)

        # check 2: does it contain metadata artefacts (ISBNs, URLs etc.)?
        if not should_filter:
            should_filter, filter_metadata = should_filter_metadata(sentence)

        if should_filter:
            if filter_metadata:
                filter_metadata['manifesto_id']= manifesto_id
                filter_metadata['filename']= name_of_manif_file
                running_log_of_manif_repairs.append(filter_metadata)

                filter_type = filter_metadata.get('filter_type', 'unknown')
                filtered_by_reason[filter_type] += 1
        else:
            sentences_i_kept.append(sentence)

    sentences= sentences_i_kept

    # & I cut any remaining all-numeric sentences
    sentences= [sent_left for sent_left in sentences_i_kept if re.search(r'[a-zA-Z]{3,}', sent_left)]

    # drop single-character tokens that aren't valid standalone words
    sentences= [sent_left for sent_left in sentences if len(sent_left) > 1 or sent_left in {'a', 'i'}]

    # + remove roman numeral list markers (e.g. "i) ", "iv) ")
    sentences= [re.sub(r'\b(?i:x|ix|iv|vii?i?|vi?|v|iii?|ii?)\)\s*', '', s) for s in sentences]

    # strip leading/trailing punctuation and whitespace
    sentences= [sent.strip(string.punctuation) for sent in sentences]
    sentences= [sent.strip() for sent in sentences if sent.strip()]

    cnt_of_headers= len(sentences_pre_filter) - len(sentences)

    if cnt_of_headers > 0:
        running_log_of_manif_repairs.append({
            'operation': 'number_header_filter_summary', 'manifesto_id': manifesto_id, 'filename': name_of_manif_file, 'total_headers_filtered': cnt_of_headers,'by_reason': dict(filtered_by_reason)})
        print(f"This notebook has now gotten shot of {garbage_count} garbage instances, w/ {cnt_of_headers} headers ({dict(filtered_by_reason)})")
    elif garbage_count > 0:
        print(f"this notebook got shot of {garbage_count} garbage sentences")

    # step 5 goes more granular yet, w/ sentence-lvl repair & record construction
    records_on_manif_sents= []

    for idx, sentence in enumerate(sentences, start=1):
        repaired_sentence, cnt_of_sents_repaired, repair_details= apply_sentence_level_repairs(sentence)
        for repair_detail in repair_details:
            repair_detail['operation']= 'sentence_repair'
            repair_detail['manifesto_id']= manifesto_id
            repair_detail['filename']= name_of_manif_file
            repair_detail['sentence_id']= idx
            running_log_of_manif_repairs.append(repair_detail)

        record_on_this_manif_sent= {'manifesto_id': manifesto_id, 'party_code': party_code, 'party_name': party_name if party_name else party_code, 'election_year': election_year,'filename': name_of_manif_file,
            'sentence_id': idx,  'sentence_text': repaired_sentence, 'repair_count': cnt_of_sents_repaired,}
        records_on_manif_sents.append(record_on_this_manif_sent)

    return records_on_manif_sents, running_log_of_manif_repairs


## Loading the PDF Manifesto Corpus

The bulk of the corpus is PDF files, named by MARPOR party code and election year. I'm iterating over them here, running each through the OCR extraction function, and storing the raw text with provenance metadata before any sentence-level processing happens.

In [ ]:
print("Loading in all the available manifestos that are pdfs (excl. excels)...")

# 1st I find all the manifestos I store as pdfs that I name w/ the naming convention of 'UK MARPOR party code prefix underscore yr'
manif_pdfs= sorted(Path(".").glob("51*.pdf")) # (if this isn't working when you're examining my work, please make sure the manifestos are in the same directory if you want to run it)
# (all manifestos from marpor start w/ 51)

# I make an exception for pdf extraction for a tricky 'We Ourselves' manifesto because its formatting was really weird, so I use a csv fallback instead
weird_we_ourselves_manif= "51210_2019.pdf"
manif_pdfs= [manif for manif in manif_pdfs if manif.name != weird_we_ourselves_manif]
print(f"\in this directory there's {len(manif_pdfs)} manifestos to extract text from (and the weird we ourselves one to do seperately).")

# Now I pullout all the manifesto text and store the metadata I need later for each manifesto
records_of_extracted_manifs= []
for manifesto_pdf_path in manif_pdfs:
    this_manifs_filename= manifesto_pdf_path.stem  # so like "51901_2019" for example
    # now I get the party code & election yr from filename
    manif_filename_match= re.match(r"(\d+)_(\d{4})", this_manifs_filename)
    if not manif_filename_match:
        print(f"didn't recognise {this_manifs_filename}, so skipping it.")
        continue
    party_code,yr= manif_filename_match.groups()

    print(f"now extracting {this_manifs_filename}...")
    text_from_this_manif= pullout_individual_manifesto_sentences_from_manifesto(manifesto_pdf_path)

    if text_from_this_manif:
        records_of_extracted_manifs.append({"party_code": party_code, "election_year": int(yr), "filename": manifesto_pdf_path.name, "raw_text": text_from_this_manif})
        print(f"Pulled out {len(text_from_this_manif):,} chars from that manifesto.")
    else:
        print("No text extracted here :(")

# Now I combine all manifestos that I extracted into a df
manifestos_df= pd.DataFrame(records_of_extracted_manifs)
manifestos_df["party_code"]= manifestos_df["party_code"].astype(str)

# & I merge w/ human-readable party names where they're available
if map_of_uk_parties_to_marpor_codes is not None:
    map_of_uk_parties_to_marpor_codes["party"]= map_of_uk_parties_to_marpor_codes["party"].astype(str)
    manifestos_df = manifestos_df.merge(map_of_uk_parties_to_marpor_codes[["party", "partyname"]], left_on="party_code", right_on="party",how="left")
    manifestos_df = manifestos_df.drop(columns=["party"])
else:
    manifestos_df["partyname"] = manifestos_df["party_code"]
print(f"\nGreat, all the extracting is done now! There were a grand total of {len(manifestos_df)} manifestos loaded in.")
print(manifestos_df[["party_code", "election_year", "filename"]].head())


## Loading the *We Ourselves* Manifesto (CSV Fallback)

The 2019 *We Ourselves* pdf comes out completely unusable from the standard OCR pipeline due to its formatting. So I don't have to drop it I'm loading its MARPOR quasi-sentence CSV as a fallback and joining it into a single text string. This only works here because there happens to be a CSV for this particular file; it's not something I can do for most of the corpus as it's often not there on the MARPOR website.

In [ ]:
# The 'We Ourselves' manifesto is loaded from its MARPOR CSV file as the PDF version produces unusable output under the standard OCR pipeline.
weird_weourselves_csv_path = "51210_201912.csv"
party_name = "We Ourselves"
print("Loading in the weirdly-formatted 'We Ourselves' manifesto from CSV now.")

if os.path.exists(weird_weourselves_csv_path):
    print(f"\nLoading it from {weird_weourselves_csv_path}")

    # I parse party code & election date from the filename as usual
    match_for_weird_weourselves= re.search(r"(\d+)_(\d{6})\.csv", weird_weourselves_csv_path)
    if match_for_weird_weourselves:
        party_code= match_for_weird_weourselves.group(1)
        date= match_for_weird_weourselves.group(2)
        election_year= int(date[:4])

        # the issue is, they're stored as quasi-sentences (the standard for when MARPOR has coded them) so I need to pull them into a single contiguous text string
        temp_weourselves_holder= pd.read_csv(weird_weourselves_csv_path, usecols=["text"])
        complete_text_from_weourselves= " ".join(temp_weourselves_holder["text"].dropna().astype(str).tolist())

        # I use the same record structure as the PDF manifesto entries
        weird_weourselves_csv_manifesto= pd.DataFrame([{"party_code": str(party_code), "election_year": election_year, "filename": weird_weourselves_csv_path, "raw_text": complete_text_from_weourselves, "partyname": party_name}])

        # & I append to the main manifesto manif df
        manifestos_df= pd.concat([manifestos_df, weird_weourselves_csv_manifesto], ignore_index=True)

        print(f"Then finally added on that last weourselves manifesto {weird_weourselves_csv_path} ({election_year})")
        print(f"It's text len is {len(complete_text_from_weourselves):,} chars")
    else:
        print(f"hmm actually couldn't load in {weird_weourselves_csv_path} so check it's in the same format and folder")
else:
    print(f"\ncouldn't actually find that {weird_weourselves_csv_path} csv manifesto, check it's there")
    print("Just going to continue without it but be aware it's not there...")

# now as a bit of post-processing I sort corpus by party code and election year for consistent ordering
manifestos_df["party_code"] = manifestos_df["party_code"].astype(str)
manifestos_df = manifestos_df.sort_values(by=["party_code", "election_year"], ascending=[True, True]).reset_index(drop=True)

print(f"\n Loaded in an overall cnt of {len(manifestos_df)} manifestos, ready for the ML modelling now!!")


## Processing All Manifestos into Sentence-Level Records

Now I've got the full corpus loaded, each manifesto goes through the sentence-level pipeline. Those functions were defined above and I've explained each one at least a little bit through comments.

In [ ]:
print("Now processing manifestos into sentences...")

big_collection_of_manif_sents= []
cumulative_manif_repairs= []# my unified operation log (covers repairs, filters, and transformations)

for idx, row in manifestos_df.iterrows():
    party_code= row['party_code']
    election_year= row['election_year']
    filename_manif_came_from= row['filename']
    raw_text= row['raw_text']
    party_name= row.get('partyname', party_code)

    print(f"\n[{idx+1}/{len(manifestos_df)}] {party_name} {election_year}")

    # so this returns (sentence_records, unified_operation_log)
    sentences, operations_i_made= process_single_manifesto(party_code=party_code, election_year=election_year,name_of_manif_file=filename_manif_came_from,raw_text=raw_text,party_name=party_name,)
    big_collection_of_manif_sents.extend(sentences)
    cumulative_manif_repairs.extend(operations_i_made)

    print(f"Done!! That manifesto has {len(sentences):,} sentences extracted from it.")

print("\n Extracion pipeline completely done now!")
print(f" Overall it pulled {len(big_collection_of_manif_sents):,} sentences across the corpus")

# Summarise operation counts across the full corpus
operation_counts = Counter(op.get('operation', 'unknown') for op in cumulative_manif_repairs)
print(f"There was a grand total of {len(cumulative_manif_repairs):,} repairs done to the corpus post-extraction.")
print("of those:")
print(f"there were {operation_counts.get('pdf_repair', 0):,} pdf repairs")
print(f"{operation_counts.get('sentence_repair', 0):,} sentence repairs")
print(f"and {operation_counts.get('sentence_filter', 0):,} sentences filtered")

# i now collect sentence records into a singular df for export
sentence_df= pd.DataFrame(big_collection_of_manif_sents)
print(f"\nthe new combined sentences df has {len(sentence_df):,} rows")


### Deduplication

A handful of sentences end up duplicated in the same manifesto, most likely because OCR re-processes overlapping page regions. I'm dropping them here using the filename and sentence text as a composite key.

In [ ]:
print(f"There were {len(sentence_df):,} before dropping duplicates")
sentence_df.drop_duplicates(subset=['filename', 'sentence_text'], inplace=True)
print(f"and after, {len(sentence_df):,}")

## Output Validation

Here I do a quick check of what the extraction actually produced & a summary of the resulting corpus.

### Output Schema

I constructed a scheme so that each row of the combined df is a single extracted sentence, with these columns:

- `manifesto_id`: party code + year (e.g. `51320_2019`)
- `party_code`: MARPOR numeric party code
- `party_name`: human-readable party name from the lookup map
- `election_year`: four-digit year
- `filename`: source PDF filename
- `sentence_id`: sequential sentence number within the manifesto
- `sentence_text`: the actual sentence after all repairs and cleaning
- `repair_count`: number of repair operations applied to this sentence

In [ ]:
print("Below is a summary of the output of all this manifesto processing: ")

if len(sentence_df) >0:
    print(f"There are {len(sentence_df):,} overall sentences in the corpora.")
    print(f"These come from {sentence_df['manifesto_id'].nunique()} different manifestos.")

    print(f"\nA breakdown for each party of sentence cnt is as follows:")
    print(sentence_df.groupby('party_name').size().sort_values(ascending=False))

    print(f"\nand by year:")
    print(sentence_df.groupby('election_year').size().sort_index())

    print(f"\nThere were a grand total of {(sentence_df['repair_count'] > 0).sum():,} sentences which needed repairs")
    print(f"and the total repairs on all those sentences were: {sentence_df['repair_count'].sum():,}")

    print("\nHere's some sample sentences:")

    # I only do 5 but feel free to increase this if you want to see more of them
    samples_for_audit= sentence_df.sample(min(5, len(sentence_df)))
    for _, row in samples_for_audit.iterrows():
        print(f"\n[{row['party_name']} {row['election_year']} - Sentence {row['sentence_id']}]")
        text = row['sentence_text']
        print(f"{text[:150]}..." if len(text) > 150 else text)
else:
    print("\nProblem, there was actually no sentences extracted, that's weird. Double-check this pipeline + uploads, something's wrong")

In [ ]:
# quick verification of that df too
if len(sentence_df) > 0:
    print("\n the shiny new corpora df is:")
    sentence_df.info()

    print("\n1st few rows from that are:")
    display(sentence_df.head(4))


## Exporting the Processed Corpus

Once the validation looks clean I export the sentence corpus and the full operation log.

In [ ]:
FULL_MANIF_CORPUS_FNAME= "manifesto_sentence_corpus.csv"
FULL_LOG_OF_MANIF_REPAIRS_FNAME= "pipeline_operations_log.jsonl"  # this is the file you read if you want to checkout all the repairs, filters & transformations my code made
if len(sentence_df) > 0:
    # I save the sentence corpus as csv 1st
    sentence_df.to_csv(FULL_MANIF_CORPUS_FNAME, index=False, encoding='utf-8')
    print(f"manifesto sentence corpus is saved @ {FULL_MANIF_CORPUS_FNAME}")
    # & then save the unified operation log as JSONL (so its one record per line)
    with open(FULL_LOG_OF_MANIF_REPAIRS_FNAME, 'w', encoding='utf-8') as repair_f:
        for operation in cumulative_manif_repairs:
            repair_f.write(json.dumps(operation, ensure_ascii=False) + '\n')
    print(f"& operation repair log is saved @ {FULL_LOG_OF_MANIF_REPAIRS_FNAME}")
    # + I save an excel copy for accessibility, just in case
    corpus_excel_file= FULL_MANIF_CORPUS_FNAME.replace('.csv', '.xlsx')
    sentence_df.to_excel(corpus_excel_file, index=False, engine='openpyxl')
    print(f"also saved an excel file on top of that csv for good measure @ {corpus_excel_file}")
    
    # and most importnatly, I generate a version-controlled filename parquet (so I can see what corpus was produced when)
    date_str= datetime.today().strftime("%Y%m%d")
    content_hash= hashlib.md5(pd.util.hash_pandas_object(sentence_df, index=True).values).hexdigest()[:12]
    parquet_filename= f"manifesto_corpus_{date_str}_{content_hash}.parquet"
    sentence_df.to_parquet(parquet_filename, index=False, engine="fastparquet")
    print(f"And most importantly for downstream analysis, saved the parquet @ {parquet_filename}")